# EXP-015 Audit (CPU)

量化当前视觉输入瓶颈。不训练、不提交、不占 GPU。


In [ ]:

import os, json, math, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

def find_root(base='/kaggle/input', depth=0, max_depth=3):
    if not os.path.isdir(base):
        return None
    try:
        entries = sorted(os.listdir(base))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(base, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'train.csv')) and os.path.isdir(os.path.join(p, 'train_series')):
            return p
        if depth < max_depth - 1:
            r = find_root(p, depth + 1, max_depth)
            if r:
                return r
    return None

ROOT = find_root()
assert ROOT, 'no competition mount'
print('ROOT', ROOT)
train = pd.read_csv(os.path.join(ROOT, 'train.csv'))
series = pd.read_csv(os.path.join(ROOT, 'train_series.csv'))
print('train', train.shape, 'series', series.shape)
print('series columns:', list(series.columns))
print('train columns:', list(train.columns))
print(series.head(3).to_string())
series.to_pickle('/kaggle/working/series_meta.pkl')
train.to_pickle('/kaggle/working/train_meta.pkl')
json.dump({'root': ROOT}, open('/kaggle/working/root.json','w'))
print('meta cached')


In [ ]:

# ========== A. Dataset-level series stats ==========
import numpy as np, pandas as pd
series = pd.read_pickle('/kaggle/working/series_meta.pkl')
s = series.copy()

# series per study (all planes)
per_study = s.groupby('StudyInstanceUID').size()
per_study_plane = s.groupby(['StudyInstanceUID','Anatomical_Plane']).size().unstack(fill_value=0)

def pct_block(name, arr):
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        print(name, 'NOT MEASURED')
        return
    qs = np.percentile(a, [25,50,75,90,95])
    print(f'{name}: n={a.size} P25={qs[0]:.3f} P50={qs[1]:.3f} P75={qs[2]:.3f} P90={qs[3]:.3f} P95={qs[4]:.3f} mean={a.mean():.3f}')

print('=== series per study (all planes) ===')
pct_block('series_per_study', per_study.values)
print('=== series per study per plane ===')
for c in per_study_plane.columns:
    pct_block(f'series_{c}', per_study_plane[c].values)
    print(f'  share studies with 0 {c}: {(per_study_plane[c]==0).mean():.3%}')

print('plane value counts:')
print(s['Anatomical_Plane'].value_counts(dropna=False))
if 'Fluid_Sensitive' in s.columns:
    print('Fluid_Sensitive value counts:')
    print(s['Fluid_Sensitive'].value_counts(dropna=False))
    fs = s.groupby(['StudyInstanceUID','Anatomical_Plane'])['Fluid_Sensitive'].agg(['sum','count'])
    print('fluid_true per study-plane:')
    pct_block('fluid_true', fs['sum'].values)

# laterality-like columns
print('laterality-like series cols:', [c for c in s.columns if any(k in c.lower() for k in ['late','side','left','right','hand','leg'])])
print('NOT MEASURED laterality in series.csv' if not any(k in c.lower() for c in s.columns for k in ['late','side','left','right']) else 'found laterality cols')


In [ ]:

# ========== B. Series selection (current rule) ==========
# rule from rsna_v3: sort Fluid_Sensitive asc, drop_duplicates(Fluid_Sensitive), head(MAX_SERIES=4)
import numpy as np, pandas as pd
series = pd.read_pickle('/kaggle/working/series_meta.pkl')
MAX_SERIES = 4

def current_select(df):
    if 'Fluid_Sensitive' in df.columns:
        return df.sort_values('Fluid_Sensitive', kind='stable').drop_duplicates(subset=['Fluid_Sensitive']).head(MAX_SERIES)
    return df.head(MAX_SERIES)

rows = []
for uid, sub in series.groupby('StudyInstanceUID'):
    for plane, praw in sub.groupby('Anatomical_Plane'):
        sel = current_select(praw)
        rows.append({
            'uid': uid, 'plane': plane,
            'n_raw': len(praw),
            'n_sel': len(sel),
            'n_drop': len(praw) - len(sel),
            'raw_fluid': int(praw['Fluid_Sensitive'].sum()) if 'Fluid_Sensitive' in praw.columns else -1,
            'sel_fluid': int(sel['Fluid_Sensitive'].sum()) if 'Fluid_Sensitive' in sel.columns else -1,
            'sel_ids': ','.join(sel['SeriesInstanceUID'].astype(str)),
            'drop_ids': ','.join(praw.loc[~praw['SeriesInstanceUID'].isin(sel['SeriesInstanceUID']), 'SeriesInstanceUID'].astype(str)),
        })
sel_df = pd.DataFrame(rows)
sel_df.to_pickle('/kaggle/working/selection.pkl')
print('study-planes', len(sel_df))
print(sel_df.groupby('plane')[['n_raw','n_sel','n_drop']].agg(['mean','median','max']))
print('share n_sel==1', (sel_df.n_sel==1).mean())
print('share n_sel<=2', (sel_df.n_sel<=2).mean())
print('share any drop', (sel_df.n_drop>0).mean())
print('share drop all but 1', (sel_df.n_drop == sel_df.n_raw-1).mean())
print('total raw', sel_df.n_raw.sum(), 'kept', sel_df.n_sel.sum(), 'dropped', sel_df.n_drop.sum())
print('info-loss ratio dropped/raw', sel_df.n_drop.sum()/sel_df.n_raw.sum())


In [ ]:

# ========== C/D. DICOM geometry: sort vs projection, coverage mm, adjacent gap ==========
import os, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom

ROOT = json.load(open('/kaggle/working/root.json'))['root']
series = pd.read_pickle('/kaggle/working/series_meta.pkl')
sel_df = pd.read_pickle('/kaggle/working/selection.pkl')
train_series_dir = os.path.join(ROOT, 'train_series')

def files_for(uid, sid):
    for c in [os.path.join(train_series_dir, uid, sid), os.path.join(train_series_dir, sid)]:
        if os.path.isdir(c):
            return [os.path.join(c,f) for f in os.listdir(c) if f.lower().endswith('.dcm')]
    return []

def slice_normal(iop):
    iop = np.asarray(iop, dtype=float).reshape(2, 3)
    n = np.cross(iop[0], iop[1])
    nn = np.linalg.norm(n)
    return n/nn if nn > 0 else None

# sample series: all study-planes if possible but cap for runtime; prefer selected series
random.seed(42)
# sample studies to keep runtime reasonable but stats solid
uids = sel_df['uid'].drop_duplicates().tolist()
sample_uids = random.sample(uids, min(60, len(uids)))
print('sampling', len(sample_uids), 'studies of', len(uids))

rec = []
for uid in sample_uids:
    sub = sel_df[sel_df.uid==uid]
    for _, r in sub.iterrows():
        plane = r.plane
        for tag, ids in [('sel', r.sel_ids), ('drop', r.drop_ids)]:
            if not ids:
                continue
            for sid in ids.split(',')[:3]:
                files = files_for(uid, sid)
                if len(files) < 2:
                    rec.append(dict(uid=uid, plane=plane, sid=sid, role=tag, n_files=len(files)))
                    continue
                ipp_z, proj, spac, orients, descs = [], [], [], [], []
                iop0 = None
                for f in files:
                    try:
                        d = pydicom.dcmread(f, stop_before_pixels=True)
                    except Exception:
                        continue
                    ipp = d.get('ImagePositionPatient')
                    iop = d.get('ImageOrientationPatient')
                    if iop is not None and iop0 is None:
                        iop0 = iop
                    if ipp is not None:
                        ipp_z.append(float(ipp[2]))
                        if iop is not None:
                            n = slice_normal(iop)
                            if n is not None:
                                proj.append(float(np.dot(np.asarray(ipp, float), n)))
                    st = d.get('SpacingBetweenSlices') or d.get('SliceThickness')
                    if st is not None:
                        spac.append(float(st))
                    if iop is not None:
                        orients.append(tuple(np.round(np.asarray(iop, float), 3)))
                    descs.append(str(d.get('SeriesDescription', '')))
                    # laterality fields
                lat_tags = []
                for f in files[:5]:
                    try:
                        d = pydicom.dcmread(f, stop_before_pixels=True)
                    except Exception:
                        continue
                    for k in ['Laterality', 'ImageLaterality', 'PatientOrientation', 'BodyPartExamined', 'PatientPosition']:
                        v = d.get(k)
                        if v is not None:
                            lat_tags.append((k, str(v)))
                z = np.asarray(ipp_z, float)
                p = np.asarray(proj, float)
                z_order = np.argsort(z) if z.size else []
                p_order = np.argsort(p) if p.size == z.size and p.size else []
                sort_diff = np.nan
                if z.size and p.size == z.size and z.size >= 2:
                    # rank correlation style: fraction of pairwise order disagreements
                    zs, ps = z[z_order], p[z_order]
                    # compare order of files by z vs by projection
                    # if we sort by z vs by p, do we get same sequence?
                    idx_z = z_order
                    idx_p = p_order
                    # hamming of relative ranks
                    rank_z = np.empty_like(idx_z); rank_z[idx_z] = np.arange(len(idx_z))
                    rank_p = np.empty_like(idx_p); rank_p[idx_p] = np.arange(len(idx_p))
                    sort_diff = float((rank_z != rank_p).mean())
                # physical span
                def span(arr):
                    if arr.size < 2: return np.nan
                    return float(arr.max() - arr.min())
                # make_clips simulation n_clips=4
                n = len(files)
                used = set()
                if n >= 3:
                    step = max(1, (n - 2) // 4)
                    for c in range(1, n - 1, step):
                        if len(used)//3 >= 4:
                            break
                        used.update({c-1, c, c+1})
                    used = set(list(used)[:12])
                raw_mm = span(z)
                if used and z.size == n:
                    uz = z[sorted(used)]
                    used_mm = span(uz)
                    cover = used_mm / raw_mm if raw_mm and raw_mm > 0 else np.nan
                    # adjacent gap in used clips (3-slice windows)
                    gaps = []
                    for c in range(1, n-1, max(1,(n-2)//4)):
                        trip = [c-1,c,c+1]
                        if max(trip) < n:
                            zz = z[trip]
                            gaps.append(float(np.max(np.diff(np.sort(zz)))))
                    # also consecutive used physical gaps
                    su = np.sort(uz)
                    all_gaps = np.diff(su) if su.size>1 else []
                else:
                    used_mm, cover, gaps, all_gaps = np.nan, np.nan, [], []
                rec.append(dict(
                    uid=uid, plane=plane, sid=sid, role=tag,
                    n_files=n,
                    z_span_mm=raw_mm,
                    proj_span_mm=span(p),
                    slice_gap_med=float(np.median(spac)) if spac else np.nan,
                    sort_disagree_frac=sort_diff,
                    used_slices=len(used),
                    used_span_mm=used_mm if 'used_mm' in dir() else np.nan,
                    cover_ratio=cover,
                    clip_trip_gap_max=float(np.max(gaps)) if gaps else np.nan,
                    used_consec_gap_med=float(np.median(all_gaps)) if len(all_gaps) else np.nan,
                    used_consec_gap_p90=float(np.percentile(all_gaps,90)) if len(all_gaps) else np.nan,
                    used_consec_gap_max=float(np.max(all_gaps)) if len(all_gaps) else np.nan,
                    n_orient=len(set(orients)),
                    series_desc=descs[0] if descs else '',
                    laterality=';'.join(f'{k}={v}' for k,v in dict(lat_tags).items()),
                ))

aud = pd.DataFrame(rec)
aud.to_csv('/kaggle/working/exp015_geometry.csv', index=False)
print('rows', len(aud), 'studies', aud.uid.nunique())
print(aud.head(3).to_string())
print('wrote exp015_geometry.csv')


In [ ]:

# ========== summarize geometry ==========
import numpy as np, pandas as pd
aud = pd.read_csv('/kaggle/working/exp015_geometry.csv')
sel = aud[aud.role=='sel']
drop = aud[aud.role=='drop']

def pct(name, a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        print(name, 'NOT MEASURED')
        return
    q = np.percentile(a, [25,50,75,90,95])
    print(f'{name}: n={a.size} P25={q[0]:.4f} P50={q[1]:.4f} P75={q[2]:.4f} P90={q[3]:.4f} P95={q[4]:.4f}')

print('=== selected series ===')
for c in ['n_files','z_span_mm','slice_gap_med','sort_disagree_frac','used_slices','cover_ratio','clip_trip_gap_max','used_consec_gap_med','used_consec_gap_p90','used_consec_gap_max']:
    pct(c, sel[c].values)

print('=== dropped series (info loss candidates) ===')
for c in ['n_files','z_span_mm','slice_gap_med','n_orient']:
    pct('drop_'+c, drop[c].values if len(drop) else [])

print('=== sorting ===')
sd = sel['sort_disagree_frac'].dropna()
print('series with sort_disagree>0:', (sd>0).mean() if len(sd) else 'NOT MEASURED', 'count', int((sd>0).sum()), '/', len(sd))
print('series with sort_disagree>=0.1:', float((sd>=0.1).mean()) if len(sd) else 'NOT MEASURED')
# studies affected
aff = sel.groupby('uid')['sort_disagree_frac'].max()
print('studies max_disagree>0:', float((aff>0).mean()) if len(aff) else 'NOT MEASURED')
print('studies max_disagree>=0.1:', float((aff>=0.1).mean()) if len(aff) else 'NOT MEASURED')

print('=== laterality ===')
lat = sel['laterality'].fillna('')
print('series with any laterality-like tag:', float((lat.str.len()>0).mean()))
print('unique laterality strings sample:')
print(lat.value_counts().head(15))
print('n_orient>1 selected:', int((sel.n_orient>1).sum()), 'of', len(sel))

print('=== cover by plane ===')
print(sel.groupby('plane')[['n_files','cover_ratio','z_span_mm','used_consec_gap_med']].median())


In [ ]:

# ========== F. Target x sampling matrix (static knowledge + measured plane presence) ==========
import numpy as np, pandas as pd
sel_df = pd.read_pickle('/kaggle/working/selection.pkl')
# plane presence
pres = sel_df.groupby('plane').size()
print('study-plane rows', pres.to_dict())

# how often a study has each plane selected >=1
uid_plane = sel_df.groupby(['uid','plane'])['n_sel'].max().unstack(fill_value=0)
print('share studies with selected series per plane:')
for c in uid_plane.columns:
    print(f'  {c}: {(uid_plane[c]>0).mean():.3%}')

matrix = [
    # target, likely plane, expected region, current coverage concern
    ("ACL", "Sagittal", "intercondylar notch / ligament", "only ~3-4 clips all planes; ligament may be missed"),
    ("MCL", "Coronal/Sagittal", "medial collateral / tibial edge", "no laterality flip check"),
    ("Medial Meniscus", "Sagittal/Coronal", "medial compartment posterior horn", "laterality + local window"),
    ("Lateral Meniscus", "Sagittal/Coronal", "lateral compartment", "laterality + local window"),
    ("Medial OA", "Coronal/Sagittal", "medial compartment cartilage", "laterality; broad coverage needed"),
    ("Lateral OA", "Coronal/Sagittal", "lateral compartment cartilage", "laterality"),
    ("PF OA", "Axial/Sagittal", "patellofemoral", "axial z_span huge (observed 90-130mm) but only 12 slices used"),
    ("Effusion", "any fluid-sensitive", "suprapatellar recess", "depends on Fluid_Sensitive selection"),
    ("Synovitis", "fluid-sensitive", "synovium / recess", "subtle; sampling quality critical"),
    ("Baker's", "Sagittal", "popliteal cyst", "posterior region may be outside mid-volume clips"),
    ("Contusion", "any / PD/T2", "bone marrow", "needs marrow-sensitive series"),
    ("Fracture", "any bone", "cortex / line", "needs bone window / full FOV"),
]
print('TARGET | PLANE | REGION | CONCERN')
for t,p,r,c in matrix:
    print(f'{t} | {p} | {r} | {c}')
print('NOTE: plane/region is expert prior, not measured lesion maps. NOT MEASURED: per-target evidence coverage in pixels.')


In [ ]:

# final machine-readable summary
import json, numpy as np, pandas as pd
aud = pd.read_csv('/kaggle/working/exp015_geometry.csv')
sel = aud[aud.role=='sel']
summary = {
    'n_studies_sampled': int(aud.uid.nunique()),
    'n_selected_series': int(len(sel)),
    'sort_disagree_share_series': float((sel.sort_disagree_frac>0).mean()) if sel.sort_disagree_frac.notna().any() else None,
    'sort_disagree_share_studies': float(sel.groupby('uid')['sort_disagree_frac'].max().gt(0).mean()) if sel.sort_disagree_frac.notna().any() else None,
    'cover_ratio_p50': float(np.nanpercentile(sel.cover_ratio, 50)) if sel.cover_ratio.notna().any() else None,
    'cover_ratio_p90': float(np.nanpercentile(sel.cover_ratio, 90)) if sel.cover_ratio.notna().any() else None,
    'used_consec_gap_p50': float(np.nanpercentile(sel.used_consec_gap_med, 50)) if sel.used_consec_gap_med.notna().any() else None,
    'used_consec_gap_p90': float(np.nanpercentile(sel.used_consec_gap_med.dropna(), 90)) if sel.used_consec_gap_med.notna().any() else None,
}
json.dump(summary, open('/kaggle/working/exp015_summary.json','w'), indent=2)
print(json.dumps(summary, indent=2))
print('AUDIT_DONE')
